# 04. Model Training and Comparison

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Where we are:** notebook 03 produced a set of features that describe a live shipment: how bad the weather is, whether the trip is already running slow, how risky the route is and how dependable the carrier is.

**Goal of this notebook:** choose the model that will sit behind the triage queue. We answer five questions.

1. Is a model better than the rule of thumb a coordinator could apply without one?
2. Which of three model types is best?
3. Can we trust the percentage it shows? If it says 80%, are about 80% of those shipments disrupted?
4. Where should the alert line sit so that false alarms stay low?
5. Does it still help on the shipments where the weather report gives no warning?

**The rule we follow:** one fifth of the shipments is set aside at the start and not touched in this notebook. Every decision here is made on the other four fifths. Notebook 05 uses the untouched fifth to check the final model once.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA, OCHRE = "#2a78d6", "#eb6834", "#1baf7a", "#a07a12"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

## 1. Load the data and build the features

`add_features` is the same function as in notebook 03.

In [2]:
clean = pd.read_csv(ROOT / "data" / "processed" / "aura_shipments.csv")

TARGET = "Disruption_Occurred"

WEATHER_SEVERITY = {"Clear": 0, "Rain": 1, "Fog": 2, "Storm": 3, "Hurricane": 4}
TYPICAL_PACE = {"Air": 2.35, "Rail": 20.9, "Road": 17.9, "Sea": 41.5}   # hours per 1,000 km in clear weather
HIGH_RISK_SCORE = 6.0        # geopolitical risk at or above this counts as a warning sign
LOW_RELIABILITY = 0.70       # carrier reliability below this counts as a warning sign


def add_features(df):
    """Add the engineered columns. Uses only inputs known during transit."""
    out = df.copy()
    severity = out["Weather_Condition"].map(WEATHER_SEVERITY)     # blank if the weather is missing or unknown
    hours = out["Lead_Time_Days"] * 24
    typical = out["Transport_Mode"].map(TYPICAL_PACE)

    out["weather_severity"] = severity                            # 0 clear ... 4 hurricane
    out["severe_weather_flag"] = (severity >= 3).astype(float).where(severity.notna())
    out["pace_h_per_1000km"] = hours / (out["Distance_km"] / 1000)
    out["pace_vs_typical"] = out["pace_h_per_1000km"] / typical    # 1.0 is normal, 3.0 is three times slower
    out["red_flag_count"] = (
        (severity >= 3).astype(int)
        + (out["Geopolitical_Risk_Score"] >= HIGH_RISK_SCORE).astype(int)
        + (out["Carrier_Reliability_Score"] < LOW_RELIABILITY).astype(int)
    )
    # Exposure of cold-chain cargo. Shown to the coordinator, never given to the model.
    out["hours_to_go"] = hours - out["Elapsed_Transit_Hours_Simulated"]
    out["cold_gap_hours"] = out["Remaining_Buffer_Hours_Simulated"] - out["hours_to_go"]
    return out


MODEL_NUMERIC = [
    "Geopolitical_Risk_Score", "Carrier_Reliability_Score", "Lead_Time_Days", "Distance_km",
    "weather_severity", "severe_weather_flag", "pace_h_per_1000km", "pace_vs_typical", "red_flag_count",
]
MODEL_CATEGORICAL = ["Weather_Condition", "Transport_Mode"]
MODEL_FEATURES = MODEL_NUMERIC + MODEL_CATEGORICAL

data = add_features(clean)
assert TARGET not in MODEL_FEATURES, "The outcome is in the feature list"

print(f"Shipments: {len(data):,}   Features: {len(MODEL_FEATURES)}   Disruption rate: {data[TARGET].mean():.1%}")

Shipments: 5,000   Features: 11   Disruption rate: 61.3%


## 2. Set aside the final test shipments

We split the shipments 80 / 20. The split keeps the same disruption rate on both sides.

- **Working set (80%):** used for everything in this notebook.
- **Test set (20%):** locked away for notebook 05.

Inside the working set we use **5-fold cross-validation**: the working set is cut into 5 equal parts, the model is trained on 4 and scored on the 5th, and this is repeated so every shipment is scored exactly once by a model that never saw it. All the scores below come from those unseen predictions.

In [3]:
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
work, test = train_test_split(data, test_size=0.20, stratify=data[TARGET], random_state=SEED)
work, test = work.reset_index(drop=True), test.reset_index(drop=True)

X, y = work[MODEL_FEATURES], work[TARGET]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"Working set : {len(work):,} shipments, {y.mean():.1%} disrupted")
print(f"Test set    : {len(test):,} shipments, {test[TARGET].mean():.1%} disrupted  (not used in this notebook)")

Working set : 4,000 shipments, 61.3% disrupted
Test set    : 1,000 shipments, 61.3% disrupted  (not used in this notebook)


## 3. The yardsticks

**Three baselines to beat**

- **"Alert on everything":** flag every shipment. It is right about 61% of the time and tells the coordinator nothing. This is why we do not use accuracy.
- **Red flag count:** count the warning signs from notebook 03 (severe weather, risk score 6 or more, carrier reliability under 0.70). A coordinator can do this in their head.
- **Weather, then risk score:** sort shipments by how bad the weather is, and inside each weather by the geopolitical risk score. This is the strongest simple rule we could find, so it is the one we call **the rule of thumb** from here on. If the model cannot beat it, the model is not worth having.

**Four scores**

| Score | Question it answers | Good value |
|---|---|---|
| **ROC AUC** | Does a disrupted shipment rank above a safe one? | 0.50 is a coin flip, 1.00 is perfect |
| **Average precision** | How clean is the top of the risk list? | Random is about 0.61, higher is better |
| **Brier score** | How close are the percentages to what happens? | Lower is better. Guessing the average rate for everything scores about 0.24 |
| **Precision and recall** | Of the alerts, how many are real? Of the disruptions, how many are caught? | Both high, but one trades against the other |

## 4. Three model types, each given a fair chance

- **Logistic regression:** adds up weighted inputs. Simple and easy to explain.
- **Random forest:** averages many decision trees, each trained on a random part of the data.
- **Gradient boosting:** builds decision trees one after another, each fixing the mistakes of the last.

Every model has settings that change how it learns. So that no model loses only because of poor settings, we try a small grid of settings for each and keep the best, judged by average precision.

In [4]:
import warnings

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_pipeline_for(model):
    """Scale the numbers, one-hot encode the text columns, then fit the model."""
    prepare = ColumnTransformer([
        ("numeric", StandardScaler(), MODEL_NUMERIC),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), MODEL_CATEGORICAL),
    ])
    return Pipeline([("prepare", prepare), ("model", model)])


candidates = {
    "Logistic regression": (
        LogisticRegression(max_iter=3000),
        {"model__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]},
    ),
    "Random forest": (
        RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        {"model__min_samples_leaf": [5, 10, 20, 40], "model__max_features": ["sqrt", 0.5]},
    ),
    "Gradient boosting": (
        GradientBoostingClassifier(random_state=SEED),
        {"model__n_estimators": [100, 200], "model__max_depth": [2, 3],
         "model__learning_rate": [0.03, 0.1], "model__subsample": [0.8, 1.0]},
    ),
}

tuned = {}
tuning_rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, (model, grid) in candidates.items():
        search = GridSearchCV(make_pipeline_for(model), grid, cv=folds, scoring="average_precision", n_jobs=-1)
        search.fit(X, y)
        tuned[name] = search.best_estimator_
        tuning_rows.append({
            "model": name,
            "settings_tried": len(search.cv_results_["params"]),
            "best_settings": {k.replace("model__", ""): v for k, v in search.best_params_.items()},
            "avg_precision_worst_setting": search.cv_results_["mean_test_score"].min(),
            "avg_precision_best_setting": search.best_score_,
        })

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(tuning_rows).round(3)

,model,settings_tried,best_settings,avg_precision_worst_setting,avg_precision_best_setting
0,Logistic regression,6,{'C': 0.3},0.898,0.901
1,Random forest,8,"{'max_features': 0.5, 'min_samples_leaf': 40}",0.893,0.899
2,Gradient boosting,16,"{'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200, 'subsample': 1.0}",0.889,0.899


**What this tells us:** look at the gap between the worst and best setting for each model. The gap is tiny for all three. The settings barely matter, which means the inputs are doing the work and no model is being held back by a poor choice.

## 5. Compare the models and the rules

Each tuned model now scores every shipment in the working set while that shipment is held out. We compare those scores with the two rules.

In [5]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

fold_list = list(folds.split(X, y))


def unseen_predictions(pipeline):
    """Disruption probability for every working-set shipment, from a model that did not train on it."""
    predictions = np.zeros(len(X))
    for train_idx, held_out_idx in fold_list:
        fitted = clone(pipeline).fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions[held_out_idx] = fitted.predict_proba(X.iloc[held_out_idx])[:, 1]
    return predictions


def fold_scores(scores, need_brier=True):
    """Score each of the 5 held-out parts separately, so we can see the spread."""
    rows = []
    for _, idx in fold_list:
        row = {"roc_auc": roc_auc_score(y.iloc[idx], scores[idx]),
               "avg_precision": average_precision_score(y.iloc[idx], scores[idx])}
        if need_brier:
            row["brier"] = brier_score_loss(y.iloc[idx], scores[idx])
        rows.append(row)
    return pd.DataFrame(rows)


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    proba = {name: unseen_predictions(pipeline) for name, pipeline in tuned.items()}

# The rules give a ranking, not a percentage, so they have no Brier score
flag_score = work["red_flag_count"].to_numpy()
rule_score = (work["weather_severity"] * 100 + work["Geopolitical_Risk_Score"]).to_numpy()   # weather first, risk score breaks ties

per_fold = {"Red flag count": fold_scores(flag_score, need_brier=False),
            "Rule of thumb (weather, then risk)": fold_scores(rule_score, need_brier=False)}
per_fold.update({name: fold_scores(p) for name, p in proba.items()})

summary = pd.DataFrame({
    name: {**scores.mean().add_suffix(""), **scores.std().add_suffix("_spread")}
    for name, scores in per_fold.items()
}).T[["roc_auc", "roc_auc_spread", "avg_precision", "avg_precision_spread", "brier", "brier_spread"]]
summary.round(4)

,roc_auc,roc_auc_spread,avg_precision,avg_precision_spread,brier,brier_spread
Red flag count,0.7400,0.0163,0.7781,0.0133,NaN,NaN
"Rule of thumb (weather, then risk)",0.8031,0.0163,0.8890,0.0081,NaN,NaN
Logistic regression,0.8308,0.0166,0.9011,0.0092,0.1625,0.0076
Random forest,0.8268,0.0142,0.8986,0.0076,0.1639,0.0055
Gradient boosting,0.8269,0.0173,0.8987,0.0094,0.1644,0.0069


In [6]:
metrics = [("roc_auc", "ROC AUC (higher is better)"), ("avg_precision", "Average precision (higher is better)"),
           ("brier", "Brier score (lower is better)")]
order = list(summary.index)

fig = make_subplots(rows=1, cols=3, subplot_titles=[m[1] for m in metrics], shared_yaxes=True, horizontal_spacing=0.05)
for col, (metric, _) in enumerate(metrics, start=1):
    part = summary[[metric, f"{metric}_spread"]].dropna()
    fig.add_trace(go.Scatter(
        x=part[metric], y=part.index, mode="markers",
        marker=dict(color=BLUE, size=11, line=dict(color=SURFACE, width=2)),
        error_x=dict(type="data", array=part[f"{metric}_spread"], color=BLUE, thickness=1.5, width=4),
        hovertemplate="%{y}<br>%{x:.3f}<extra></extra>",
    ), row=1, col=col)
fig.update_yaxes(categoryorder="array", categoryarray=order[::-1])
fig.update_annotations(font=dict(size=12, color=INK))
style(fig, "Models compared on unseen shipments (dot = average of 5 parts, bar = spread)", height=380)
fig.update_layout(showlegend=False, margin=dict(l=230))
fig.show()

In [7]:
from sklearn.metrics import precision_recall_curve

fig = go.Figure()
STEP = 8  # plot every 8th point to keep the notebook file small
precision, recall, _ = precision_recall_curve(y, rule_score)
fig.add_trace(go.Scatter(x=recall[::2], y=precision[::2], mode="lines", name="Rule of thumb",
                         line=dict(color=MUTED, width=2, dash="dash"),
                         hovertemplate="Rule of thumb<br>Recall %{x:.0%}<br>Precision %{y:.0%}<extra></extra>"))
for (name, p), color in zip(proba.items(), [BLUE, ORANGE, AQUA]):
    precision, recall, _ = precision_recall_curve(y, p)
    fig.add_trace(go.Scatter(x=recall[::STEP], y=precision[::STEP], mode="lines", name=name, line=dict(color=color, width=2),
                             hovertemplate=name + "<br>Recall %{x:.0%}<br>Precision %{y:.0%}<extra></extra>"))
fig.add_hline(y=y.mean(), line=dict(color=GRID, width=1))
fig.add_annotation(x=0.01, y=y.mean(), text="Alert on everything", showarrow=False, xanchor="left", yanchor="bottom",
                   font=dict(color=MUTED))
fig.update_xaxes(title_text="Recall: share of disruptions caught", tickformat=".0%", range=[0, 1])
fig.update_yaxes(title_text="Precision: share of alerts that are real", tickformat=".0%", range=[0.5, 1.03])
style(fig, "Precision against recall on unseen shipments", height=500)
fig.update_layout(legend=dict(orientation="h", y=-0.2, x=0, font=dict(color=INK)), margin=dict(b=100))
fig.show()

**How to read the curve:** move from left to right and the alert line is lowered, so more disruptions are caught (recall goes up) but more alerts are false (precision goes down). A better model has a curve that stays higher for longer.

**What this tells us:**

- **All three models beat the rule of thumb, by a small margin.** Average precision is about 0.90 for the models against 0.89 for the rule, and ROC AUC is 0.83 against 0.80. The gap in average precision is roughly one and a half times the round-to-round spread, so it is real but not dramatic.
- **The red flag count is well behind** (ROC AUC 0.74). It has only four possible values and treats a hurricane the same as a storm. It is easy to say out loud, and it is not good enough to rank a queue.
- **The first third of disruptions is easy for everyone.** Up to about 30% recall, every curve sits at 100% precision. Those are the hurricane shipments, and the rule of thumb finds them as well as any model.
- **The model earns its place further along the curve**, once the hurricanes are used up and the weather alone stops separating shipments. Section 10 measures this directly.
- **The three models are close to tied.** Their dots overlap within the spread on every score.

## 6. Can we trust the percentages?

The app shows a number such as "82% disruption risk" and sorts shipments into tiers. That only helps if the number is honest. We group the unseen predictions by the percentage the model gave, then check how many in each group really were disrupted. A trustworthy model sits on the dashed line.

In [8]:
def reliability(p, bins=(0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0001)):
    """Average predicted risk against the share really disrupted, inside each band of predictions."""
    band = pd.cut(p, bins=list(bins), right=False)
    table = pd.DataFrame({"predicted": p, "disrupted": y.to_numpy()}).groupby(band, observed=True)
    return table.agg(predicted=("predicted", "mean"), observed=("disrupted", "mean"), shipments=("disrupted", "size"))


fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfectly honest",
                         line=dict(color=MUTED, width=2, dash="dash"), hoverinfo="skip"))
for (name, p), color in zip(proba.items(), [BLUE, ORANGE, AQUA]):
    table = reliability(p)
    fig.add_trace(go.Scatter(
        x=table["predicted"], y=table["observed"], mode="lines+markers", name=name,
        line=dict(color=color, width=2), marker=dict(size=8, line=dict(color=SURFACE, width=2)),
        customdata=table["shipments"],
        hovertemplate=name + "<br>Model said %{x:.0%}<br>Really disrupted %{y:.0%}<br>Shipments: %{customdata:,}<extra></extra>",
    ))
fig.update_xaxes(title_text="Risk the model predicted", tickformat=".0%", range=[0, 1.02])
fig.update_yaxes(title_text="Share really disrupted", tickformat=".0%", range=[0, 1.02])
style(fig, "Predicted risk against what happened", height=500)
fig.update_layout(legend=dict(orientation="h", y=-0.2, x=0, font=dict(color=INK)), margin=dict(b=100))
fig.show()

gap = pd.DataFrame({
    "average_gap_between_predicted_and_observed": {
        name: reliability(p).eval("abs(predicted - observed) * shipments").sum() / len(y) for name, p in proba.items()},
    "largest_gap_in_any_band": {
        name: reliability(p).eval("abs(predicted - observed)").max() for name, p in proba.items()},
})
display(gap.round(3))
pd.concat({name: reliability(p) for name, p in proba.items()}, axis=1).round(2)

,average_gap_between_predicted_and_observed,largest_gap_in_any_band
Logistic regression,0.020,0.065
Random forest,0.013,0.028
Gradient boosting,0.013,0.030


Logistic regression                    Random forest                    Gradient boosting                   
                     predicted observed shipments     predicted observed shipments         predicted observed shipments
[0.0, 0.2)                0.17     0.15       232          0.15     0.15       189              0.16     0.15       181
[0.2, 0.3)                0.25     0.28       445          0.26     0.26       398              0.25     0.26       348
[0.3, 0.4)                0.35     0.37       464          0.35     0.38       457              0.35     0.34       524
[0.4, 0.5)                0.45     0.45       415          0.45     0.43       506              0.45     0.44       477
[0.5, 0.6)                0.55     0.54       441          0.55     0.56       576              0.55     0.54       506
[0.6, 0.7)                0.65     0.64       440          0.65     0.62       459              0.65     0.64       480
[0.7, 0.8)                0.74     0.68       330          0.75     0.76       194              0.74     0.71       280
[0.8, 0.9)                0.85     0.88       261          0.86     0.88       218              0.85     0.88       204
[0.9, 1.0)                0.98     0.99       972          0.99     0.98      1003              0.97     0.98      1000

**What this tells us:**

- **All three models are honest to within a couple of points.** On average the percentage shown is 1 to 2 points away from what happened.
- **Random forest and gradient boosting are slightly closer** (1.3 points on average, never more than 3 in any band).
- **Logistic regression is 2 points off on average, with one weak band.** When it says 70 to 80%, about 68% of those shipments are disrupted, a gap of about 6 points. Everywhere else it is within 3.
- **The two ends are the most reliable.** When a model says over 90%, 98 or 99% are disrupted. When it says under 20%, about 15% are.

## 7. Choose the model

We choose with a rule written down before looking at the answer:

1. Highest average precision wins, because the coordinator works from the top of the list.
2. If two models are closer than their round-to-round spread, they are tied on step 1, and the one with the lower Brier score wins, because the app shows the percentage on screen.

In [9]:
MODEL_NAMES = list(tuned)
model_summary = summary.loc[MODEL_NAMES]
leader = model_summary["avg_precision"].idxmax()
tie_margin = model_summary.loc[leader, "avg_precision_spread"]
tied = model_summary[model_summary["avg_precision"] >= model_summary.loc[leader, "avg_precision"] - tie_margin]

CHOSEN = tied["brier"].idxmin()
chosen_proba = proba[CHOSEN]

print(f"Highest average precision      : {leader} ({model_summary.loc[leader, 'avg_precision']:.3f})")
print(f"Within one spread of the leader: {list(tied.index)}")
print(f"Lowest Brier score among those : {CHOSEN} ({model_summary.loc[CHOSEN, 'brier']:.4f})")
print()
print(f"Chosen model: {CHOSEN}")
print("Settings    :", {k: v for k, v in tuned[CHOSEN].named_steps['model'].get_params().items()
                         if k in {'C', 'n_estimators', 'max_depth', 'learning_rate', 'subsample', 'min_samples_leaf', 'max_features'}})

Highest average precision      : Logistic regression (0.901)
Within one spread of the leader: ['Logistic regression', 'Random forest', 'Gradient boosting']
Lowest Brier score among those : Logistic regression (0.1625)

Chosen model: Logistic regression
Settings    : {'C': 0.3}


**What this tells us:**

- All three models are tied on average precision once the spread is taken into account, so the choice falls to the Brier score, and **logistic regression** wins it.
- We should be plain about how narrow this is. The three Brier scores differ by less than their own spread, and section 6 showed the tree models are a touch closer on the percentages. Any of the three would have been a reasonable choice.
- When the scores cannot separate the models, the simplest one is the right pick, and here the rule and that principle agree. Logistic regression gives each input one weight: bad weather adds this much risk, a reliable carrier takes that much away. A coordinator can follow that reasoning, and the app can show it.
- It also behaves predictably when an input changes. Improve one input and the risk moves smoothly in one direction, with no sudden jumps. That matters for the what-if simulator in notebook 05.

## 8. Do the risk tiers mean what they say?

The app sorts shipments into four tiers by predicted risk: Low (up to 30%), Medium (31 to 60%), High (61 to 80%) and Critical (above 80%). Each tier should contain clearly more disruptions than the one below it.

In [10]:
TIER_EDGES = [0, 0.30, 0.60, 0.80, 1.0001]
TIER_NAMES = ["Low (0 to 30%)", "Medium (31 to 60%)", "High (61 to 80%)", "Critical (above 80%)"]

tier = pd.cut(chosen_proba, bins=TIER_EDGES, labels=TIER_NAMES, right=False)
tiers = pd.DataFrame({"disrupted": y.to_numpy()}).groupby(tier, observed=False).agg(
    shipments=("disrupted", "size"), disrupted=("disrupted", "sum"), disruption_rate=("disrupted", "mean"))
tiers["share_of_all_shipments"] = tiers["shipments"] / tiers["shipments"].sum()
tiers["share_of_all_disruptions"] = tiers["disrupted"] / tiers["disrupted"].sum()

fig = go.Figure(go.Bar(
    x=tiers.index.astype(str), y=tiers["disruption_rate"], marker_color=BLUE,
    text=[f"{r:.0%}<br>{n:,} shipments" for r, n in zip(tiers["disruption_rate"], tiers["shipments"])],
    textposition="outside", textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{x}<br>Really disrupted: %{y:.1%}<extra></extra>",
))
fig.update_yaxes(title_text="Share really disrupted", tickformat=".0%", range=[0, 1.22])
style(fig, f"What really happened in each risk tier ({CHOSEN})", height=440)
fig.update_layout(bargap=0.4)
fig.show()

tiers.style.format({"disruption_rate": "{:.1%}", "share_of_all_shipments": "{:.1%}", "share_of_all_disruptions": "{:.1%}",
                    "shipments": "{:,}", "disrupted": "{:,}"})

,shipments,disrupted,disruption_rate,share_of_all_shipments,share_of_all_disruptions
Low (0 to 30%),677,159,23.5%,16.9%,6.5%
Medium (31 to 60%),"1,320",595,45.1%,33.0%,24.3%
High (61 to 80%),770,504,65.5%,19.2%,20.6%
Critical (above 80%),"1,233","1,192",96.7%,30.8%,48.7%


**What this tells us:**

- **The tiers separate cleanly.** About 24% of Low shipments are disrupted, 45% of Medium, 66% of High and 97% of Critical. Each step up is a clear jump, and the real rate sits inside each tier's stated range.
- **Critical means critical.** Of 1,233 shipments in the top tier, 1,192 were disrupted. A coordinator can act on that tier without second-guessing it. It holds almost half of all disruptions.
- **The limit:** Low does not mean safe. About 1 Low shipment in 4 is still disrupted, and the Low and Medium tiers together hold about 3 in 10 of all disruptions. In this file even a shipment with clear weather, a calm route and a good carrier runs into trouble fairly often. No input available at dispatch points to those cases, so no model could flag them without flagging nearly everything.

## 9. Where to draw the alert line

The model gives every shipment a percentage. The coordinator needs a line: above it, the shipment appears in the queue as needing action.

- A **low line** catches more disruptions but sends more false alarms. Too many false alarms and people stop trusting the queue.
- A **high line** means nearly every alert is real, but more disruptions slip through.

The table shows the trade for several lines, per 1,000 shipments. The last column estimates money, using three assumptions that we state openly and that the app lets the user change:

- Acting on an alert costs **$500**, the highest fee in the action table in notebook 05 (refrigerated ground transport).
- A disruption costs **10%** of the cargo value on average. A disruption is a delay or a scare far more often than a total loss.
- Acting in time avoids that cost **70%** of the time.

In [11]:
ACTION_COST_USD = 500        # assumption: the highest fee in the action table (notebook 05)
LOSS_SHARE = 0.10            # assumption: share of cargo value a disruption costs on average
ACTION_SUCCESS_RATE = 0.70   # assumption: share of interventions that avoid the cost

value = work["Cargo_Value_USD_Simulated"].to_numpy()
disrupted_mask = y.to_numpy() == 1
per_thousand = 1000 / len(y)


def alert_line(threshold, scores=chosen_proba):
    flagged = scores >= threshold
    caught = flagged & disrupted_mask
    saved = (value[caught].sum() * LOSS_SHARE * ACTION_SUCCESS_RATE - flagged.sum() * ACTION_COST_USD) * per_thousand
    return {
        "alert_line": f"{threshold:.0%}",
        "alerts_per_1000": flagged.sum() * per_thousand,
        "real_per_1000": caught.sum() * per_thousand,
        "false_alarms_per_1000": (flagged & ~disrupted_mask).sum() * per_thousand,
        "precision": caught.sum() / max(flagged.sum(), 1),
        "recall": caught.sum() / disrupted_mask.sum(),
        "net_saving_usd_per_1000": saved,
    }


lines = pd.DataFrame([alert_line(t) for t in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]])
lines.style.format({
    "alerts_per_1000": "{:.0f}", "real_per_1000": "{:.0f}", "false_alarms_per_1000": "{:.0f}",
    "precision": "{:.0%}", "recall": "{:.0%}", "net_saving_usd_per_1000": "${:,.0f}",
}).hide(axis="index")

alert_line,alerts_per_1000,real_per_1000,false_alarms_per_1000,precision,recall,net_saving_usd_per_1000
30%,831,573,258,69%,94%,"$36,717,452"
40%,715,530,185,74%,87%,"$34,049,317"
50%,611,484,127,79%,79%,"$30,917,138"
60%,501,424,77,85%,69%,"$27,114,734"
70%,391,354,37,91%,58%,"$22,372,009"
80%,308,298,10,97%,49%,"$19,194,741"
90%,243,241,2,99%,39%,"$15,283,989"


In [12]:
ALERT_LINE = 0.60

grid = np.linspace(0.10, 0.98, 89)
curve = pd.DataFrame([alert_line(t) for t in grid])

fig = go.Figure()
fig.add_trace(go.Scatter(x=grid, y=curve["precision"], mode="lines", name="Precision: share of alerts that are real",
                         line=dict(color=BLUE, width=2), hovertemplate="Line at %{x:.0%}<br>Precision %{y:.0%}<extra></extra>"))
fig.add_trace(go.Scatter(x=grid, y=curve["recall"], mode="lines", name="Recall: share of disruptions caught",
                         line=dict(color=ORANGE, width=2), hovertemplate="Line at %{x:.0%}<br>Recall %{y:.0%}<extra></extra>"))
fig.add_vline(x=ALERT_LINE, line=dict(color=INK, width=1, dash="dash"))
fig.add_annotation(x=ALERT_LINE, y=1.06, text=f"Chosen alert line: {ALERT_LINE:.0%}", showarrow=False,
                   xanchor="left", font=dict(color=INK))
fig.update_xaxes(title_text="Alert line (predicted risk)", tickformat=".0%")
fig.update_yaxes(tickformat=".0%", range=[0, 1.1])
style(fig, f"What moving the alert line does ({CHOSEN})", height=460)
fig.update_layout(legend=dict(orientation="h", y=-0.22, x=0, font=dict(color=INK)), margin=dict(b=100))
fig.show()

chosen_line = alert_line(ALERT_LINE)
print(f"At an alert line of {ALERT_LINE:.0%}, per 1,000 shipments:")
print(f"  alerts raised      : {chosen_line['alerts_per_1000']:.0f}")
print(f"  of which real      : {chosen_line['real_per_1000']:.0f}  (precision {chosen_line['precision']:.0%})")
print(f"  false alarms       : {chosen_line['false_alarms_per_1000']:.0f}")
print(f"  disruptions caught : {chosen_line['recall']:.0%} of all disruptions")
print(f"  'alert on everything' would raise 1,000 alerts with {(~disrupted_mask).mean() * 1000:.0f} false alarms")

At an alert line of 60%, per 1,000 shipments:
  alerts raised      : 501
  of which real      : 424  (precision 85%)
  false alarms       : 77
  disruptions caught : 69% of all disruptions
  'alert on everything' would raise 1,000 alerts with 388 false alarms


**Decision: the alert line is 60%.**

- **It matches the tiers.** 60% is where Medium ends and High begins, so the rule for the queue is simple to say: everything High and Critical needs a look.
- **False alarms drop by four fifths.** Flagging every shipment gives 388 false alarms per 1,000. At the 60% line the queue holds about 500 alerts, 85% of them real, with 77 false alarms. It still catches about 69% of all disruptions.
- **Why not 50%?** At 50% precision and recall are both 79%. It catches 60 more disruptions per 1,000 and adds 50 false alarms. That is a fair trade for a team with spare capacity. We start at 60% because trust in the queue is hard to win back once it is lost.
- **What the money column says, stated fairly.** Under our assumptions a lower line always saves more, because the cargo is worth far more than the $500 it costs to act. The money cannot choose the line. Followed blindly it says "act on everything", and at the 30% line the queue would hold 83% of all shipments, which is not a queue. A person's attention is the scarce resource, so we set the line for trust and let the ranking do the rest.
- **Read the dollar figures as a comparison between lines, not a forecast.** They are large because the simulated cargo values are whole container loads, some worth several million dollars.
- **The line is not a wall.** Shipments under 60% stay visible further down the queue, ranked by risk, cargo value and urgency. The app can expose the line as a slider so a team with more capacity can lower it.

## 10. Is the model better than the rule of thumb where it counts?

A fair test gives both the same workload. We let the rule of thumb raise exactly as many alerts as the model does at the chosen line, taking the worst weather first and the highest risk score inside each weather, and compare how many of those alerts are real.

In [13]:
model_flag = chosen_proba >= ALERT_LINE
n_alerts = int(model_flag.sum())
rule_flag = np.zeros(len(y), dtype=bool)
rule_flag[np.argsort(-rule_score, kind="stable")[:n_alerts]] = True   # the n_alerts shipments the rule ranks highest


def alert_quality(flag, label):
    caught = flag & disrupted_mask
    return {
        "method": label, "alerts": int(flag.sum()), "real": int(caught.sum()),
        "false_alarms": int((flag & ~disrupted_mask).sum()),
        "precision": caught.sum() / flag.sum(), "recall": caught.sum() / disrupted_mask.sum(),
    }


head_to_head = pd.DataFrame([alert_quality(rule_flag, "Rule of thumb"), alert_quality(model_flag, CHOSEN)]).set_index("method")
fewer = 1 - head_to_head.loc[CHOSEN, "false_alarms"] / head_to_head.loc["Rule of thumb", "false_alarms"]
print(f"With the same number of alerts ({n_alerts:,}), the model raises {fewer:.0%} fewer false alarms than the rule of thumb.")
print(f"Alerts the two lists share: {(rule_flag & model_flag).sum():,} of {n_alerts:,}")
head_to_head.style.format({"precision": "{:.1%}", "recall": "{:.1%}", "alerts": "{:,}", "real": "{:,}", "false_alarms": "{:,}"})

With the same number of alerts (2,003), the model raises 5% fewer false alarms than the rule of thumb.
Alerts the two lists share: 1,814 of 2,003


,alerts,real,false_alarms,precision,recall
method,,,,,
Rule of thumb,"2,003","1,679",324,83.8%,68.5%
Logistic regression,"2,003","1,696",307,84.7%,69.2%


### The harder test: shipments the weather report says nothing about

Notebook 02 found that every hurricane shipment is disrupted. Anyone can call those. Here we score two smaller groups:

- **No hurricane:** everything except the certain cases.
- **Calm weather:** clear, rain or fog only. Here the weather gives no strong warning, so the model has to work from the route, the carrier and the pace.

In [14]:
weather = work["Weather_Condition"]
groups = {
    "All shipments": np.ones(len(work), dtype=bool),
    "No hurricane": (weather != "Hurricane").to_numpy(),
    "Calm weather (clear, rain, fog)": weather.isin(["Clear", "Rain", "Fog"]).to_numpy(),
}

rows = []
for group, mask in groups.items():
    for label, scores in [("Rule of thumb", rule_score), (CHOSEN, chosen_proba)]:
        rows.append({
            "group": group, "method": label, "shipments": int(mask.sum()), "disruption_rate": y[mask].mean(),
            "roc_auc": roc_auc_score(y[mask], scores[mask]),
            "avg_precision": average_precision_score(y[mask], scores[mask]),
        })
harder = pd.DataFrame(rows).set_index(["group", "method"])

calm = groups["Calm weather (clear, rain, fog)"]
calm_flag = model_flag & calm
safe_flag = (chosen_proba < 0.30) & calm
print(f"Calm-weather shipments                      : {calm.sum():,}, of which {y[calm].mean():.1%} are disrupted")
print(f"  alerts the model raises among them        : {calm_flag.sum():,}, of which {y[calm_flag].mean():.0%} are real")
print(f"  shipments it rates Low (under 30%)        : {safe_flag.sum():,}, of which {y[safe_flag].mean():.0%} are disrupted")
harder.style.format({"shipments": "{:,}", "disruption_rate": "{:.1%}", "roc_auc": "{:.3f}", "avg_precision": "{:.3f}"})

Calm-weather shipments                      : 2,407, of which 42.4% are disrupted
  alerts the model raises among them        : 444, of which 64% are real
  shipments it rates Low (under 30%)        : 677, of which 23% are disrupted


**What this tells us:**

- **Same workload, slightly better alerts.** With an identical 2,003 alerts across the working set, the model's list holds 1,696 real disruptions against 1,679 for the rule of thumb. That is 17 more caught and 5% fewer false alarms. The two lists share 9 alerts in 10. This is a small edge, and we should not present it as a large one.
- **The edge grows as the weather stops helping.** With hurricanes removed, the model's ROC AUC is 0.75 against 0.71 for the rule. In calm weather it is 0.68 against 0.61. The rule has almost nothing to go on there, and the model still has the carrier and the pace.
- **In calm weather the model's alerts are right 64% of the time**, against a base rate of 42% for those shipments. And the 677 shipments it rates Low are disrupted 23% of the time, about half the base rate. So it both finds trouble and clears shipments where the weather report is silent.
- **The limit:** a ROC AUC of 0.68 in calm weather is modest. The model gives a coordinator a real head start on shipments that look fine, and it is far from seeing everything.

## What we found and what happens next

### Answers to the five questions

| # | Question | Answer |
|---|---|---|
| 1 | Is a model better than the rule of thumb? | Yes, by a small margin. ROC AUC 0.83 against 0.80, and 5% fewer false alarms for the same number of alerts |
| 2 | Which model type is best? | Logistic regression, narrowly. All three are tied within the spread, and it is the simplest |
| 3 | Can we trust the percentage? | Yes, to within about 2 points on average. Its weakest band (70 to 80%) reads about 6 points high |
| 4 | Where is the alert line? | 60%. About 85% of alerts are real, and about 69% of disruptions are caught |
| 5 | Does it help when the weather gives no warning? | Yes, with limits. In calm weather its alerts are right 64% of the time against a base rate of 42% |

### Decision carried into notebook 05

- **Model:** logistic regression with C = 0.3
- **Features:** the 11 columns in `MODEL_FEATURES`
- **Alert line:** 60%
- **Risk tiers:** Low up to 30%, Medium 31 to 60%, High 61 to 80%, Critical above 80%

### Next

Notebook 05 trains the chosen model on the whole working set, scores it once on the untouched test set, checks how it copes with noisy and missing input, and saves it for the app.

### Limits we state openly

- Model settings were chosen and scored on the same five parts of the working set, so the scores here are slightly flattering. The untouched test set in notebook 05 gives the fair number.
- The model's lead over a two-column rule of thumb is small. Most of what can be predicted in this file comes from the weather, and a rule can read the weather too. What the model adds is an honest percentage, a ranking inside each weather, and a backup when the weather field is missing.
- The money figures rest on three assumptions (intervention cost, share of value a disruption costs, success rate) and on simulated cargo values. They are there to compare alert lines, not to forecast a budget.
- The scores describe this dataset, which looks generated. Shipments from a real carrier could score differently, and the model should be re-checked on them before use.